CASO PRATICO - RECENSIONI IMDB CON TF-IDF

In questo esercizio mettiamo insieme più tecniche viste.
recensione -> testo -> TF-IDF -> vettore numerico -> classificatore -> positiva/negativa

Sentimen analysis su larga scala.
il dataset ha 50.000 opinioni con opinioni recensioni e tanto rumore digitali

- gestione di un dataset di grandi dimensioni
- chirurgia del vocabolario potando le parole innutili
- valutiamo la capacità di generalizzazione

Dataset IMDB
Lavorare con 50.000 recensioni è una sfida di scala, è impossibile senza un metodo.
Il dataset è sporco come il web reale, dobbiamo trattare questi dati con rispetto, pulirli e caricarli con rispetto, assicurandoci che il nostro computer rimanga senza ram durante il processo.

Il primo passo è assicurarci che i nostri campioni siano giusti.

Campionamento e Bilancio
Come per capire se un risotto è cotto non è necessario mangiarlo tutto, anche con il dataset la stessa cosa, campioniamo per verificare la qualità dei dati.

Garantire l'equità del segnale
Mescoliamo il dataset per garantire che nel nostro cucchiaio di test ci sia la stessa proporzione di recensioni positive/negative che sono presenti nel db originale.
Solo così la matematica sarà equa.

Ma prima della matematica serve una pulizia profonda dei dati
Integrità dei Dati
I dati provenienti dal web sono sporchi, dobbiamo rimuovere tag html e residui markup (es. br) usando le espressioni regolari.
E' fondamentale poi, garantire encoding UTF-8 per gestire correttamente caratteri speciali o accenti presenti nelle recensioni internazionali.
Mescolare poi i dati è fondamentale per evitare che l'ordine di lettura (es. tutte le recensioni positive delle negative) comprometta la discesa del gradiente (shuffling). Per evitare che il modello impari l'ordine invece del senso.

Una volta puliti i dati dobbiamo decidere come dividerli per lo studio
Logica di Partizione
Training vs Testing
In un dataset di 50.000 campioni, una divisione classica prevede l'uso dell'80% per l'addestramento e il 20% per il test finale. Questo permette di avere un numero sufficiente di esempi per l'apprendimento e una validazione statisticamente significativa. Come nella vita serve un esame finale (con il 20% dei dati di test che rimane chiuso in una cassaforte fino al giorno dell'esame)
L'obbiettivo è minimizzare l'errore di generalizzazione misurando la discrepanza tra le performace sui due set distinti.

Ora che la strategia è chiara, affrontiamo il nemico numero 1, l'infinità delle parole
Ottimizzazione del Vocabolario
Ridurre la sparsità senza perdere significato.
In un dataset di grandi dimensioni, il numero di parole uniche può superare facilmente le centinaia di migliaia. Creare una matrice TF-IDF su tutte queste parole porterebbe alla 'curse of dimensionality' e a problemi di memoria insormontabili.
La maggir parte delle parolo sono solo rumore di fondo, se tenessimo ogni singolo refuso o parola rarissima creeremmo una matrice così grande da essere inutilizzabile, dobbiamo tenere solo ciò che serve. 
Dobbiamo selezionare solo le feature più informative, Parametri come 'min_df' e 'max_df' ci permettono di ignorare parole troppo rare (spesso refusi) o troppo comuni (che non aiutano a distinguere), ottimizzando drasticamente l'efficienza del modello.

Per farlo useremo dei filtri statistici molto efficienti
Parametri di Filtraggio
Potare l'albero delle parole
Il TF-IDF vectorizer è il nostro setaccio con min_df (se una paraola appare troppo poco) e max_df (per parole troppo comuni) come una sorta di stoword
Con gli n-gram permettiamo al modello di capire che 'non è buono' è diverso da 'è buono'
Così abbiamo un briciolo di comprensione del contesto locale

L'efficienza alla fine è eleganza
Analisi della Sparsità
Limitare il vocbolario a 5.000 o 10.000 termini più frequenti è spesso sufficiente per mantenere un'accuratezza elevata riducendo la complessità
Useremo poi anche il sublinear TF, l'uso di scaling logaritmico per la frequenza dei termini, questo impedisce che parole ripetute troppe volte in una singola recensione dominino il vettore. Esempio un utente che scrive 10 volte bello in un post faccia impazzire i pesi del modello.
In alcuni casi, è più utile sapere sole se una parola è presente (1) o assente (0) piuttosto che quante volte appare, formato binario (binary TF)

L'ottimizzazione del vocabolario è la nostra difesa contro over fitting, se lasciassimo al modello troppe parole specifiche, lui inizierebbe a memorizzare e non saprebbe più convergere con dati nuovi.
Un vocabolario troppo specifico tenderà a memorizzare le singole recensioni invece di comprendere i pattern generali del linguaggio positivo o negativo, termini che esprimono le emozioni.
La dimensione dello spazio delle feature influenza direttamente la stabilità dei coefficienti del modello di classificazione.

E poi una volta addestrato il nostro cervello digitale è ora di metterlo alla prova
Valutazioe della Generalizzazione
Il modello funziona nel mondo reale?
Una volta addestrato il modello sull'80% dei dati di IMDb, la vera sfida è osservare come si comporta sul restanto 20%. Un modello che ottiene il 99% di accuratezza sul training ma solo il 70% sul test è affetto da overfitting, è come un pilota che sa guidare solo sul simulatore e va nel panico in pista.
Vediamo come interpretare le metriche di valutazione per capire se il nostro classificatore ha imparato il 'senso' dei sentiment cinematografico o se ha solo trovato scorciatoie statistiche legate a nomi di attori o registi specifici

Vediamo gli strumenti professionali per misurare questa robustezza
Robustezza e Bias
Oltre i numeri della validation
- Test set Inviolabile: i dati di test non devono mai essere usati durante l'ottimizzazione dei parametri o la selezione del vocabolario
- Cross-Validation: suddividere il training set in più pieghe per garantire che le performance non dipendano da un singolo taglio fortunato dei dati. Dividiamo il nostro materiale di studio in 5  (esempio) parti e di fare 5 esami diversi
- Analisi degli errori: esaminare manualmente le recensioni classificate erroneamente per identificare sarcasmo o negazioni complesse.
- L'errore atteso su nuovi campioni (generalizzazione) è la misura definitiva del successo di un progetto di Deep Learning classico.
Capire il fallimento è l'unico modo per migliorare la nostra pipeline

Ma non guardiamo solo i numeri, guardiamo anche l'errore
Monitoraggio delle Metriche
- F1-Score Bilanciato: anche se l'IMDb è bilanciato, monitorare l'f1-score assicura che il modello non stia privileggiando una classe a discapito dell'altra in termini di precisione
- Matrice di Confusione: visualizzare quanti film 'capolavoro' vengono scambiati per 'disastri' ci dice molto sulla sensibilità del vocabolario ottimizzato
- Learning Curves: osservare come migliorano le performance all'aumentare dei dati di training ci aiuta a capire se abbiamo bisogno di più esempi o di un modello più complesso.

Ma la prova finale è il test live
Validazione Esterna
Predizione su testi inediti.
Il test finale consiste nel fornire al modello frasi scritte da noi in questo momento (frasi ambigue e piene di sfumature). Se il modello riconosce correttamente il sentiment di una recensione inedita, significa che il processo di vettorizzazione, TF-IDF, ha estratto il segnale corretto; e che la nostra rete neurale ha compreso la struttura del sentimento.
La probabilità di errore su un nuovo campione 'x' dipende dalla bontà dell'approssimazione della funzione di decisione reale.

In [1]:
"""
================================================================================
Sentiment Analysis su Scala Reale con Keras 3 e PyTorch (Best Practices 2026)
================================================================================
Questo script mostra rappresenta una pipeline completa di Deep Learning per la 
Sentiment Analysis, partendo dal caricamento del dataset IMDb (50.000 recensioni)
fino alla creazione di una rete neurale densa utilizzando le ultime funzionalità
di Keras 3 con backend PyTorch.

Concetti chiave:
- Gestione Big Data e pulizia chirurgica del testo.
- Vettorizzazione TF-IDF ottimizzata per ridurre l'overfitting.
- Architettura Keras Funzionale per massima flessibilità.
- Strategie di regolarizzazione moderne (Dropout, Batch Normalization).
================================================================================
"""

import os

# Impostiamo il backend di Keras su PyTorch prima di importare Keras (Best Practice 2026)
os.environ["KERAS_BACKEND"] = "torch"
# Cruciale su Windows per evitare deadlock con i Transformers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import numpy as np
import pandas as pd
import torch
import keras
from keras import layers, models, ops
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from transformers import pipeline # Per il confronto con lo Stato dell'Arte
import kagglehub # Utilizzato per il download efficiente del dataset

def scarica_e_carica_dati():
    """
    Scarica il dataset IMDb da Kaggle e lo carica in un DataFrame Pandas.
    
    Returns:
        pd.DataFrame: DataFrame contenente le recensioni e i relativi sentiment.
    """
    print("[1/6] Scaricamento del dataset in corso...")
    # Scarichiamo il dataset ufficiale IMDb di 50k recensioni
    # Nota: Nel 2026 l'efficienza nel caricamento dati è prioritaria
    path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
    csv_path = os.path.join(path, "IMDB Dataset.csv")
    
    # Carichiamo i dati assicurandoci dell'encoding UTF-8 (Slide 5)
    df = pd.read_csv(csv_path, encoding='utf-8')
    
    # Mappiamo le etichette testuali in valori numerici (0 = Negativo, 1 = Positivo)
    df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
    
    return df

def pulizia_testo_chirurgica(text):
    """
    Esegue una pulizia profonda del testo rimuovendo tag HTML e rumore digitale.
    
    Args:
        text (str): Testo grezzo della recensione.
        
    Returns:
        str: Testo pulito e normalizzato.
    """
    # 1. Rimozione Tag HTML (Slide 5: <br />, ecc.) usando le espressioni regolari
    text = re.sub(r'<.*?>', ' ', text)
    
    # 2. Rimozione di caratteri non alfabetici e normalizzazione in minuscolo
    # Questo riduce il rumore e la dimensionalità del vocabolario (Slide 7)
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    
    # 3. Rimozione di spazi bianchi superflui all'inizio e alla fine
    return text.strip()

def preprocessa_dataset(df):
    """
    Preprocessa l'intero dataset: pulizia, splitting e vettorizzazione TF-IDF.
    
    Args:
        df (pd.DataFrame): DataFrame originale.
        
    Returns:
        tuple: (X_train, X_test, y_train, y_test, vectorizer)
    """
    print("[2/6] Pulizia dei testi e Stratified Splitting...")
    # Applichiamo la pulizia a tutte le 50.000 recensioni
    df['review_cleaned'] = df['review'].apply(pulizia_testo_chirurgica)
    
    # Suddivisione Stratificata 80/20 (Slide 4 e 6)
    # Garantiamo che il bilanciamento 50/50 sia mantenuto in entrambi i set
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        df['review_cleaned'], 
        df['sentiment'], 
        test_size=0.20, 
        stratify=df['sentiment'], 
        random_state=42,
        shuffle=True # Fondamentale per evitare bias sull'ordine (Slide 5)
    )
    
    print("[3/6] Vettorizzazione TF-IDF con filtri statistici (Slide 8 & 9)...")
    # Configuriamo il TfidfVectorizer con le best practices discusse
    vectorizer = TfidfVectorizer(
        max_features=10000,   # Limitiamo a 10.000 termini per evitare 'maledizione dimensionalità'
        min_df=5,             # Scartiamo errori di battitura (parole che appaiono in < 5 doc)
        max_df=0.7,           # Scartiamo stop-words dinamiche (parole presenti in > 70% doc)
        ngram_range=(1, 2),    # Catturiamo il contesto locale (es: "non buono")
        sublinear_tf=True     # Applichiamo logaritmo alla frequenza (Slide 9)
    )
    
    # Trasformiamo i testi in matrici sparse
    X_train = vectorizer.fit_transform(X_train_raw).toarray()
    X_test = vectorizer.transform(X_test_raw).toarray()
    
    # In Keras 3 con backend PyTorch, le label per la classificazione binaria 
    # devono spesso essere in formato 2D (N, 1) per evitare errori nelle metriche
    y_train_2d = np.array(y_train).reshape(-1, 1)
    y_test_2d = np.array(y_test).reshape(-1, 1)
    
    return X_train, X_test, y_train_2d, y_test_2d, vectorizer, X_test_raw

def crea_modello_funzionale(input_dim):
    """
    Costruisce una rete neurale densa usando la Functional API di Keras 3.
    
    Args:
        input_dim (int): Numero di feature in ingresso (Max Features del TF-IDF).
        
    Returns:
        keras.Model: Il modello compilato.
    """
    print("[4/6] Costruzione del cervello digitale (Keras Functional API)...")
    
    # Definizione dell'Input Layer
    inputs = layers.Input(shape=(input_dim,), name="tfidf_input")
    
    # Primo blocco denso con Regolarizzazione (Slide 10)
    # Batch Normalization aiuta la stabilità del gradiente
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x) # Dropout al 50% per combattere l'overfitting
    
    # Secondo blocco denso più stretto (architettura a imbuto)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output Layer per classificazione binaria
    outputs = layers.Dense(1, activation='sigmoid', name="sentiment_output")(x)
    
    # Creazione del modello
    model = models.Model(inputs=inputs, outputs=outputs, name="IMDb_Sentiment_Analyzer")
    
    # Compilazione con ottimizzatore AdamW (standard industriale 2026)
    model.compile(
        optimizer="adamw",
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.F1Score(average="micro", threshold=0.5)]
    )
    
    return model

def main():
    # 1. Caricamento Dati
    df = scarica_e_carica_dati()
    
    # 2. Preprocessing e Vettorizzazione
    X_train, X_test, y_train, y_test, vectorizer, X_test_raw = preprocessa_dataset(df)
    
    # 3. Creazione Modello
    input_dim = X_train.shape[1]
    model = crea_modello_funzionale(input_dim)
    model.summary()
    
    # 4. Addestramento con Early Stopping (Best Practice per evitare Overfitting)
    print("\n[5/6] Addestramento del modello...")
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=3, 
        restore_best_weights=True
    )
    
    history = model.fit(
        X_train, y_train,
        validation_split=0.2, # Usiamo una parte del training per la validazione interna
        epochs=20,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # 5. Valutazione della Generalizzazione (Slide 11 & 12)
    print("\n[6/6] Valutazione della Generalizzazione (F1-Comparison)...")
    
    # Per un confronto equo e veloce (i Transformers sono pesanti su CPU), 
    # selezioniamo un campione di 500 recensioni dal Test Set
    sample_size = 500 
    X_test_sample_raw = X_test_raw.iloc[:sample_size]
    y_test_sample = y_test[:sample_size]
    
    # 1. Predizioni Modello Keras (Custom)
    X_test_sample_tfidf = vectorizer.transform(X_test_sample_raw).toarray()
    keras_preds_prob = model.predict(X_test_sample_tfidf, verbose=0)
    keras_preds = (keras_preds_prob > 0.5).astype(int).flatten()
    # Appiattiamo y_test_sample per passarlo a sklearn
    keras_f1 = f1_score(y_test_sample.flatten(), keras_preds)
    
    # 2. Predizioni Hugging Face (SOTA)
    print(f"Inizializzazione Hugging Face (SOTA) con framework PyTorch...")
    device = 0 if torch.cuda.is_available() else -1
    hf_pipeline = pipeline(
        "sentiment-analysis", 
        model="distilbert-base-uncased-finetuned-sst-2-english", 
        device=device, 
        framework="pt"
    )
    
    print(f"Calcolo predizioni Hugging Face su {sample_size} campioni...")
    hf_preds = []
    # Processiamo uno alla volta per massima stabilità su Windows (come nel file reference)
    for testo in tqdm(X_test_sample_raw.tolist(), desc="Analisi HF"):
        # Troncamento a 1000 caratteri per evitare sforamento 512 token
        res = hf_pipeline(testo[:1000], truncation=True)[0]
        hf_preds.append(1 if res['label'] == 'POSITIVE' else 0)
    
    hf_f1 = f1_score(y_test_sample.flatten(), hf_preds)
    
    print("-" * 50)
    print("RISULTATI CONFRONTO STATISTICO (F1-SCORE)")
    print("-" * 50)
    print(f"KERNAS CUSTOM (TF-IDF): {keras_f1:.4f}")
    print(f"HUGGING FACE (SOTA):    {hf_f1:.4f}")
    print("-" * 50)
    if hf_f1 > keras_f1:
        print(f"Risultato: Hugging Face vince di {(hf_f1 - keras_f1):.4f} punti!")
    else:
        print("Risultato: Il tuo modello custom tiene testa allo stato dell'arte!")
    
    # 7. Test Manuale Live e Confronto (Slide 14)
    test_frasi = [
        "This movie was an absolute masterpiece of modern cinema!",
        "The plot was boring and the acting was terrible.",
        "Not as good as the first one, but still worth a watch.",
        "Despite the rain, the film was a ray of sunshine." # Test sul contesto (Slide 14)
    ]
    
    print("\n--- SFIDA: Keras Personalizzato vs Hugging Face Pre-trained ---")
    for frase in test_frasi:
        # Analisi con Modello Keras (TF-IDF)
        frase_pulita = pulizia_testo_chirurgica(frase)
        vettore = vectorizer.transform([frase_pulita]).toarray()
        pred_keras = model.predict(vettore, verbose=0)[0][0]
        label_keras = "POSITIVO" if pred_keras > 0.5 else "NEGATIVO"
        conf_keras = pred_keras if pred_keras > 0.5 else 1 - pred_keras
        
        # Analisi con Hugging Face (Transformer)
        res_hf = hf_pipeline(frase)[0]
        label_hf = res_hf['label']
        conf_hf = res_hf['score']
        
        print(f"Frase: '{frase}'")
        print(f"  > KERAS (TF-IDF):   {label_keras} ({conf_keras:.2%})")
        print(f"  > HUGGING FACE:     {label_hf} ({conf_hf:.2%})")
        print("-" * 20)

if __name__ == "__main__":
    main()

# ==============================================================================
# SPIEGAZIONE DETTAGLIATA:
#
# 1. Impostazione Backend: 'os.environ["KERAS_BACKEND"] = "torch"' dice a Keras 
#    di usare PyTorch per il calcolo dei tensori invece di TensorFlow o JAX.
#
# 2. Pulizia: La Regex r'<.*?>' identifica tutto ciò che sta tra parentesi 
#    angolari (tag HTML) e lo sostituisce con uno spazio. Fondamentale per IMDb.
#
# 3. TF-IDF: Usiamo 'sublinear_tf=True' perché l'importanza di una parola non 
#    cresce in modo lineare con la sua frequenza (Slide 9). I 'Bigrams' permettono
#    di capire espressioni composte da due parole.
#
# 4. Keras Functional API: Rispetto al modello 'Sequential', l'API Funzionale 
#    specifica esplicitamente i collegamenti tra i layer (es. outputs=outputs(x)).
#    È lo standard richiesto per architetture complesse e multi-input.
#
# 5. Regolarizzazione: 'BatchNormalization' normalizza l'output di un layer per
#    velocizzare l'addestramento. 'Dropout' spegne casualmente dei neuroni per
#    costringere la rete a non memorizzare i dati (evitando l'Overfitting).
#
# 7. Hugging Face vs Keras: Mentre il TF-IDF osserva la statistica delle parole
#    nel nostro dataset specifico (veloce ma limitato), i Transformers di 
#    Hugging Face usano il 'Transfer Learning'. Hanno una comprensione 
#    profonda della grammatica e del sarcasmo, rendendoli spesso più 
#    precisi su frasi complesse ma più pesanti computazionalmente.
# ==============================================================================

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/6] Scaricamento del dataset in corso...


100%|██████████| 25.7M/25.7M [00:05<00:00, 5.25MB/s]

Extracting files...


[2/6] Pulizia dei testi e Stratified Splitting...
[3/6] Vettorizzazione TF-IDF con filtri statistici (Slide 8 & 9)...
[4/6] Costruzione del cervello digitale (Keras Functional API)...


...


[5/6] Addestramento del modello...
Epoch 1/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.8585 - f1_score: 0.8585 - loss: 0.3257 - val_accuracy: 0.8959 - val_f1_score: 0.8957 - val_loss: 0.3146
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - accuracy: 0.9227 - f1_score: 0.9228 - loss: 0.1978 - val_accuracy: 0.8867 - val_f1_score: 0.8900 - val_loss: 0.2729
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.9476 - f1_score: 0.9476 - loss: 0.1370 - val_accuracy: 0.8844 - val_f1_score: 0.8840 - val_loss: 0.3158
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9644 - f1_score: 0.9643 - loss: 0.0959 - val_accuracy: 0.8856 - val_f1_score: 0.8887 - val_loss: 0.3636
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - accuracy: 0.9730 - f1_score: 0.9730 - loss: 0.0752 - val_accuracy: 0.8799 - val_f1_score: 0.8845 - val_loss: 0.3837

[6/6] Valutazione della Generalizzazione (F1-Comparison)...
Inizializzazione Hugging Face (SOTA) con fr

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5110.87it/s]


Calcolo predizioni Hugging Face su 500 campioni...


Analisi HF: 100%|██████████| 500/500 [00:05<00:00, 83.77it/s]


--------------------------------------------------
RISULTATI CONFRONTO STATISTICO (F1-SCORE)
--------------------------------------------------
KERNAS CUSTOM (TF-IDF): 0.9028
HUGGING FACE (SOTA):    0.8211
--------------------------------------------------
Risultato: Il tuo modello custom tiene testa allo stato dell'arte!

--- SFIDA: Keras Personalizzato vs Hugging Face Pre-trained ---
Frase: 'This movie was an absolute masterpiece of modern cinema!'
  > KERAS (TF-IDF):   POSITIVO (98.66%)
  > HUGGING FACE:     POSITIVE (99.98%)
--------------------
Frase: 'The plot was boring and the acting was terrible.'
  > KERAS (TF-IDF):   NEGATIVO (100.00%)
  > HUGGING FACE:     NEGATIVE (99.98%)
--------------------
Frase: 'Not as good as the first one, but still worth a watch.'
  > KERAS (TF-IDF):   POSITIVO (90.41%)
  > HUGGING FACE:     POSITIVE (99.96%)
--------------------
Frase: 'Despite the rain, the film was a ray of sunshine.'
  > KERAS (TF-IDF):   NEGATIVO (74.55%)
  > HUGGING FACE:   